In [16]:
import os
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis

def extract_stats(signal):
    return {
        'max': np.max(signal),
        'min': np.min(signal),
        'mean': np.mean(signal),
        'sd': np.std(signal),
        'rms': np.sqrt(np.mean(signal**2)),
        'skewness': skew(signal),
        'kurtosis': kurtosis(signal),
        'crest': np.max(np.abs(signal)) / np.sqrt(np.mean(signal**2)),
        'form': np.sqrt(np.mean(signal**2)) / np.mean(np.abs(signal))
    }

channel_names = ['tachometer', 'underhang_axial', 'underhang_radial', 'underhang_tangential',
                  'overhang_axial', 'overhang_radial', 'overhang_tangential', 'microphone']

window_size = 2048
rows = []
base_path = r"C:\Users\hp\predictive-maintenance-diagnosis\data\raw\MAFAULDA"

fault_folders = {
    'normal': 'normal',
    'horizontal_misalignment': 'horizontal-misalignment',
    'imbalance': 'imbalance',
    'vertical_misalignment': 'vertical-misalignment',
    'underhang_bearing': 'underhang',
    'overhang_bearing': 'overhang',
}

for fault_label, folder_name in fault_folders.items():
    folder_path = os.path.join(base_path, folder_name)
    file_count = 0

    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith('.csv'):
                filepath = os.path.join(root, file)
                try:
                    data = pd.read_csv(filepath, header=None).values
                except Exception as e:
                    print(f"Skipped {filepath}: {e}")
                    continue

                if data.shape[1] != 8:
                    print(f"Unexpected shape {data.shape} in {filepath}, skipping")
                    continue

                n_windows = data.shape[0] // window_size

                for w in range(n_windows):
                    window = data[w*window_size : (w+1)*window_size]
                    row = {}
                    for ch_idx, ch_name in enumerate(channel_names):
                        stats = extract_stats(window[:, ch_idx])
                        for stat_name, value in stats.items():
                            row[f'{ch_name}_{stat_name}'] = value
                    row['fault_type'] = fault_label
                    row['file_id'] = filepath
                    rows.append(row)

                file_count += 1
                if file_count % 10 == 0:
                    print(f"{fault_label}: {file_count} files processed so far...")

    print(f"{fault_label}: DONE, total {file_count} files processed")

df = pd.DataFrame(rows)
print(df.shape)
print(df['fault_type'].value_counts())

output_path = r"C:\Users\hp\predictive-maintenance-diagnosis\data\processed\mafaulda_features_FINAL.csv"

df.to_csv(output_path, index=False)
print(f"Dataset saved successfully to:\n{output_path}")

normal: 10 files processed so far...
normal: 20 files processed so far...
normal: 30 files processed so far...
normal: 40 files processed so far...
normal: 50 files processed so far...
normal: 60 files processed so far...
normal: 70 files processed so far...
normal: 80 files processed so far...
normal: 90 files processed so far...
normal: DONE, total 98 files processed
horizontal_misalignment: 10 files processed so far...
horizontal_misalignment: 20 files processed so far...
horizontal_misalignment: 30 files processed so far...
horizontal_misalignment: 40 files processed so far...
horizontal_misalignment: 50 files processed so far...
horizontal_misalignment: 60 files processed so far...
horizontal_misalignment: 70 files processed so far...
horizontal_misalignment: 80 files processed so far...
horizontal_misalignment: 90 files processed so far...
horizontal_misalignment: 100 files processed so far...
horizontal_misalignment: 110 files processed so far...
horizontal_misalignment: 120 fil